In [1]:
import numpy as np
import qim3d

In [2]:
# Generate synthetic blob
vol = qim3d.generate.volume2(noise_scale = 0.02,)


qim3d.viz.volumetric(vol, grid_visible=True)

Output()

In [3]:
# Generate tubular synthetic blob
vol = qim3d.generate.volume2(base_shape = (10, 300, 300),
                        final_shape = (100, 100, 100),
                        noise_scale = 0.3,
                        threshold = 0.0,
                        shape = "cylinder",
                        decay_rate=1,
                        gamma=0.3
                        )

# Visualize synthetic volume
qim3d.viz.volumetric(vol, grid_visible=True)

Output()

In [4]:
# Generate tubular synthetic blob
vol = qim3d.generate.volume2(base_shape = (200, 100, 100),
                              final_shape = (400,100,100),
                              noise_scale = 0.03,
                              threshold = 0.85,
                              decay_rate=20,
                              gamma=0.15,
                              shape = "tube",
                              tube_hole_ratio = 0.4,
                              )

# Visualize synthetic volume
qim3d.viz.volumetric(vol, grid_visible=True)

Output()

In [ ]:
import qim3d

# Generate synthetic collection of volumes
num_volumes = 15
volume_collection, labels = qim3d.generate.volume_collection(num_volumes=num_volumes)

# Visualize the collection
qim3d.viz.volumetric(volume_collection, grid_visible=True)

In [ ]:
import qim3d
# Generate synthetic collection of dense objects
vol, labels = qim3d.generate.volume_collection(
    value_range = (255, 255),
    noise_range = (0.03, 0.04),
    threshold_range = (0.99, 0.99),
    gamma_range = (0.02, 0.02),
    decay_rate_range = (10,10)
    )

# Visualize the collection
qim3d.viz.volumetric(vol)

In [ ]:
# Generate synthetic collection of cylindrical structures
volume_collection, labels = qim3d.generate.volume_collection(
    num_volumes = 40,
    collection_shape = (300, 150, 150),
    shape_range = ((280, 10, 10), (290, 15, 15)),
    noise_range = (0.06,0.09),
    rotation_degree_range = (0,5),
    threshold_range = (0.1,0.3),
    gamma_range = (0.10, 0.20),
    shape = "cylinder"
    )

# Visualize the collection
qim3d.viz.volumetric(volume_collection)

In [ ]:
# Generate synthetic collection of tubular (hollow) structures
volume_collection, labels = qim3d.generate.volume_collection(
    num_volumes = 10,
    collection_shape = (200, 200, 200),
    shape_range = ((185,35,35), (190,45,45)),
    noise_range = (0.02, 0.03),
    rotation_degree_range = (0,5),
    threshold_range = (0.6, 0.7),
    gamma_range = (0.1, 0.11),
    shape = "tube",
    tube_hole_ratio = 0.15,
    )

# Visualize the collection
qim3d.viz.volumetric(volume_collection)

In [1]:
import k3d
import ipywidgets as widgets
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
from qim3d.utils._misc import scale_to_float16
from qim3d.generate import _noise, _distances, _shape_noise, _threshold, _tube_fade

In [11]:
class VolumeVisualizer:
    def __init__(self, base_shape=(128,128,128), seed=0, initial_config=None):
        self.base_shape = base_shape
        self.seed = seed
        self.noise_type = 'perlin'
        self.shape = None
        self.axis = 0
        self.max_value = 255

        self.config = {
            'noise_scale': 0.02,
            'decay_rate': 10,
            'gamma': 1.0,
            'threshold': 0.5,
            'tube_hole_ratio': 0.5
        }
        if initial_config:
            self.config.update(initial_config)

        self.state = {}
        self._build_widgets()
        self._setup_plot()
        self._display_ui()

    def _generate_noise(self):
        self.noise, self.center, self.z, self.y, self.x = _noise(
            self.base_shape, self.config['noise_scale'], self.noise_type, self.seed
        )

    def _generate_state(self):
        scaled_distance = _distances(self.z, self.y, self.x, self.center, self.shape, self.axis)
        faded_distance = np.power(scaled_distance, self.config['decay_rate'])
        vol_normalized = _shape_noise(faded_distance, self.noise)
        self.state.update(dict(
            scaled_distance=scaled_distance,
            faded_distance=faded_distance,
            vol_normalized=vol_normalized
        ))

    def _compute_volume(self):
        vol_gamma = np.power(self.state["vol_normalized"], self.config['gamma'])
        vol_thresh = _threshold(vol_gamma, self.max_value, self.config['threshold'])
        vol = _tube_fade(vol_thresh, self.shape, self.axis, self.config['tube_hole_ratio'])
        return scale_to_float16(vol)

    def _build_widgets(self):
        self.noise_slider = widgets.FloatSlider(value=self.config['noise_scale'], min=0, max=0.2, step=0.001, description='Noise')
        self.decay_slider = widgets.FloatSlider(value=self.config['decay_rate'], min=0.1, max=20, step=0.1, description='Decay')
        self.gamma_slider = widgets.FloatSlider(value=self.config['gamma'], min=0.1, max=2.0, step=0.1, description='Gamma')
        self.threshold_slider = widgets.FloatSlider(value=self.config['threshold'], min=0.0, max=1.0, step=0.05, description='Threshold')
        self.noise_type_dropdown = widgets.Dropdown(options=['perlin', 'simplex'], value='perlin', description='Noise Type')
        self.shape_dropdown = widgets.Dropdown(options=[None, 'cylinder', 'tube'], value=None, description='Shape')
        self.tube_hole_ratio_slider = widgets.FloatSlider(value=self.config['tube_hole_ratio'], min=0.0, max=1.0, step=0.05, description='Tube hole ratio')

        # Observers
        self.noise_slider.observe(self._on_noise_change, names='value')
        self.noise_type_dropdown.observe(self._on_noise_change, names='value')
        self.decay_slider.observe(self._on_decay_change, names='value')
        self.gamma_slider.observe(self._on_gamma_or_thresh_change, names='value')
        self.threshold_slider.observe(self._on_gamma_or_thresh_change, names='value')
        self.shape_dropdown.observe(self._on_shape_change, names='value')
        self.tube_hole_ratio_slider.observe(self._on_tube_change, names='value')

    def _setup_plot(self):
        self._generate_noise()
        self._generate_state()
        vol = self._compute_volume()

        cmap = plt.get_cmap('magma')
        attr_vals = np.linspace(0.0, 1.0, num=cmap.N)
        rgb_vals = cmap(np.arange(0, cmap.N))[:, :3]
        color_map = np.column_stack((attr_vals, rgb_vals)).tolist()

        pixel_count = np.prod(vol.shape)
        y1, x1 = 256, 16777216
        y2, x2 = 32, 134217728
        a = (y1 - y2) / (x1 - x2)
        b = y1 - a * x1
        samples = int(min(max(a * pixel_count + b, 64), 512))

        self.plot = k3d.plot()
        self.plt_volume = k3d.volume(
            vol,
            bounds=[0, vol.shape[2], 0, vol.shape[1], 0, vol.shape[0]],
            color_map=color_map,
            samples=samples,
            color_range=[np.min(vol), np.max(vol)],
            opacity_function=[],
            interpolation=True,
        )
        self.plot += self.plt_volume

    # ==== Observers ====

    def _on_noise_change(self, change=None):
        self.config['noise_scale'] = self.noise_slider.value
        self.noise_type = self.noise_type_dropdown.value
        self._generate_noise()
        self._generate_state()
        self._update_volume()

    def _on_decay_change(self, change=None):
        self.config['decay_rate'] = self.decay_slider.value
        scaled_distance = _distances(self.z, self.y, self.x, self.center, self.shape, self.axis)
        faded_distance = np.power(scaled_distance, self.config['decay_rate'])
        vol_normalized = _shape_noise(faded_distance, self.noise)
        self.state.update(dict(
            scaled_distance=scaled_distance,
            faded_distance=faded_distance,
            vol_normalized=vol_normalized
        ))
        self._update_volume()

    def _on_gamma_or_thresh_change(self, change=None):
        self.config['gamma'] = self.gamma_slider.value
        self.config['threshold'] = self.threshold_slider.value
        self._update_volume()

    def _on_shape_change(self, change=None):
        self.shape = self.shape_dropdown.value
        self._generate_state()
        self._update_volume()

    def _on_tube_change(self, change=None):
        self.config['tube_hole_ratio'] = self.tube_hole_ratio_slider.value
        if self.shape == 'tube':
            self._update_volume()

    def _update_volume(self):
        new_vol = self._compute_volume()
        self.plt_volume.volume = new_vol

    def _display_ui(self):
        controls = widgets.VBox([
            self.noise_type_dropdown,
            self.noise_slider,
            self.decay_slider,
            self.gamma_slider,
            self.threshold_slider,
            self.shape_dropdown,
            self.tube_hole_ratio_slider
        ])
        display(controls)
        display(self.plot)


In [12]:
config = {'gamma':1.4}
viz = VolumeVisualizer(base_shape=(100,100,100), seed=2, initial_config=config)


Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

9.8
0.1
0.30000000000000004
0.4
0.6
1.0
0.7999999999999999
1.0
2.9
3.1
5.8999999999999995


In [1]:
import qim3d
seed=42

volume, labels, placed_locs = qim3d.generate.volume_collection(
        num_volumes = 10,
        collection_shape = (300, 150, 150),
        positions = None,
        shape_range = ((180, 6, 6), (250, 10, 10)),
        noise_range = (0.0, 0.000001),
        rotation_degree_range = (0,5),
        threshold_range = (0.7,0.9),
        gamma_range = (0.20,0.21),
        shape = "cylinder", 
        seed = seed, 
        return_positions = True, 
        verbose = False
        )

Objects placed:   0%|          | 0/10 [00:00<?, ?it/s]

In [2]:

volume_noisy, labels_noisy = qim3d.generate.volume_collection(
        num_volumes = 10,
        collection_shape = (300, 150, 150),
        positions = placed_locs,
        shape_range = ((180, 6, 6), (250, 10, 10)),
        noise_range = (0.08, 0.1),
        rotation_degree_range = (0,5),
        threshold_range = (0.7,0.9),
        gamma_range = (0.20,0.21),
        shape = "cylinder", 
        return_positions = False, 
        verbose = False,
        seed=seed
        )

Objects placed:   0%|          | 0/10 [00:00<?, ?it/s]